# Hermite polynomials with memory, look-ahead, and online correction

This notebook extends the symbolic exercises in `SymPlay.ipynb` (execution counts 48–75). It uses the **physicists' Hermite polynomials**

$$H_n(x)=(-1)^n e^{x^2}\frac{d^n}{dx^n}e^{-x^2},\qquad
e^{2xt-t^2}=\sum_{n=0}^{\infty}H_n(x)\frac{t^n}{n!}.$$

The central idea mirrors pencil-and-paper thought: retain earlier terms, answer the requested order, and prepare a configurable number of future terms. Exact recurrence is the production method; an online statistical model makes speculative forecasts and confidence intervals, then learns from exact corrections.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from functools import wraps
from pathlib import Path
from time import perf_counter
import json

import numpy as np
import sympy as sp
from IPython.display import display, Markdown

sp.init_printing()
x, t = sp.symbols('x t', real=True)
print(f"SymPy {sp.__version__}; NumPy {np.__version__}")

SymPy 1.14.0; NumPy 2.4.1


## 1. Original Rodrigues exercise, generated rather than handwritten

The next cell duplicates the original construction of unevaluated Rodrigues expressions and their evaluated forms for orders 0–10.

In [2]:
def rodrigues(n: int, evaluate: bool = True) -> sp.Expr:
    if not isinstance(n, int) or n < 0:
        raise ValueError("n must be a non-negative integer")
    expr = (-1)**n * sp.exp(x**2) * sp.Derivative(sp.exp(-x**2), (x, n))
    return sp.expand(expr.doit()) if evaluate else expr

original_rodrigues = {n: rodrigues(n, evaluate=False) for n in range(11)}
original_expanded = {n: rodrigues(n) for n in range(11)}

for n in range(11):
    display(Markdown(f"**$H_{{{n}}}(x)$**"))
    display(sp.Eq(sp.Symbol(f'H_{n}'), original_expanded[n]))

**$H_{0}(x)$**

**$H_{1}(x)$**

**$H_{2}(x)$**

**$H_{3}(x)$**

**$H_{4}(x)$**

...

**$H_{6}(x)$**

**$H_{7}(x)$**

**$H_{8}(x)$**

**$H_{9}(x)$**

**$H_{10}(x)$**

## 2. An exact engine with memory and look-ahead

The recurrence

$$H_{n+1}(x)=2xH_n(x)-2nH_{n-1}(x)$$

needs only the last two terms, avoiding repeated high-order differentiation. `around(n)` returns surrounding orders; `get(n)` automatically computes several future terms so the cache stays ahead of demand. A decorator records timing and cache behavior. The cache can be exported for a later scheduled run.

In [3]:
def observed(method):
    # Decorator: attach elapsed time and cache-growth telemetry.
    @wraps(method)
    def wrapper(self, *args, **kwargs):
        before = len(self._cache)
        start = perf_counter()
        result = method(self, *args, **kwargs)
        self.last_call = {
            "method": method.__name__, "elapsed_ms": 1000 * (perf_counter() - start),
            "cache_before": before, "cache_after": len(self._cache)
        }
        return result
    return wrapper

@dataclass
class HermiteLookAhead:
    lookahead: int = 3
    cache_path: Path | None = None
    _cache: dict[int, sp.Expr] = field(default_factory=lambda: {0: sp.Integer(1), 1: 2*x})
    last_call: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.lookahead < 0:
            raise ValueError("lookahead must be non-negative")
        if self.cache_path is not None:
            self.cache_path = Path(self.cache_path)
            self.load()

    def _extend_to(self, target: int) -> None:
        if target < 0:
            raise ValueError("order must be non-negative")
        for k in range(max(self._cache) + 1, target + 1):
            self._cache[k] = sp.Poly(2*x*self._cache[k-1] - 2*(k-1)*self._cache[k-2], x).as_expr()

    @observed
    def get(self, n: int, prefetch: bool = True) -> sp.Expr:
        if not isinstance(n, int) or n < 0:
            raise ValueError("n must be a non-negative integer")
        self._extend_to(n + (self.lookahead if prefetch else 0))
        return self._cache[n]

    def around(self, n: int, behind: int = 2, ahead: int | None = None) -> dict[int, sp.Expr]:
        ahead = self.lookahead if ahead is None else ahead
        if min(n, behind, ahead) < 0:
            raise ValueError("n, behind, and ahead must be non-negative")
        self._extend_to(n + ahead)
        return {k: self._cache[k] for k in range(max(0, n-behind), n+ahead+1)}

    def generator_series(self, order: int) -> sp.Expr:
        self._extend_to(order)
        return sp.Add(*(self._cache[n] * t**n / sp.factorial(n) for n in range(order+1)))

    def save(self) -> None:
        if self.cache_path is None:
            return
        payload = {str(n): sp.srepr(p) for n, p in self._cache.items()}
        self.cache_path.write_text(json.dumps(payload, indent=2))

    def load(self) -> None:
        if self.cache_path is None or not self.cache_path.exists():
            return
        allowed = {"Symbol": sp.Symbol, "Integer": sp.Integer, "Add": sp.Add, "Mul": sp.Mul, "Pow": sp.Pow}
        raw = json.loads(self.cache_path.read_text())
        self._cache.update({int(n): sp.sympify(expr, locals=allowed) for n, expr in raw.items()})

engine = HermiteLookAhead(lookahead=3, cache_path=Path("hermite_polynomial_cache.json"))
H10 = engine.get(10)
print("H_10(x) =", H10)
print("telemetry:", engine.last_call)
print("cached orders:", min(engine._cache), "through", max(engine._cache))

H_10(x) = 1024*x**10 - 23040*x**8 + 161280*x**6 - 403200*x**4 + 302400*x**2 - 30240
telemetry: {'method': 'get', 'elapsed_ms': 8.536708017345518, 'cache_before': 2, 'cache_after': 14}
cached orders: 0 through 13


In [4]:
# The requested “surrounding n” view: two remembered terms and three anticipated terms.
window = engine.around(10, behind=2, ahead=3)
for n, polynomial in window.items():
    marker = "  <-- requested" if n == 10 else ""
    print(f"H_{n}(x) = {polynomial}{marker}")

H_8(x) = 256*x**8 - 3584*x**6 + 13440*x**4 - 13440*x**2 + 1680
H_9(x) = 512*x**9 - 9216*x**7 + 48384*x**5 - 80640*x**3 + 30240*x
H_10(x) = 1024*x**10 - 23040*x**8 + 161280*x**6 - 403200*x**4 + 302400*x**2 - 30240  <-- requested
H_11(x) = 2048*x**11 - 56320*x**9 + 506880*x**7 - 1774080*x**5 + 2217600*x**3 - 665280*x
H_12(x) = 4096*x**12 - 135168*x**10 + 1520640*x**8 - 7096320*x**6 + 13305600*x**4 - 7983360*x**2 + 665280
H_13(x) = 8192*x**13 - 319488*x**11 + 4392960*x**9 - 26357760*x**7 + 69189120*x**5 - 69189120*x**3 + 17297280*x


## 3. Generator-function phase from the original notebook

The original `Herm10` is the Taylor polynomial in **t** assembled from $H_n(x)/n!$. The separate `Hermonomial10` is the tenth partial sum of the exponential series after substituting $2xt-t^2$; it is not the same truncation operation because powers of $2xt-t^2$ mix degrees in $t$. Both are reproduced and compared to the exact generator.

In [5]:
Herm10 = sp.expand(engine.generator_series(10))
generator = sp.exp(2*x*t - t**2)
Hermonomial10 = sp.Add(*((2*x*t - t**2)**k / sp.factorial(k) for k in range(11)))

print("Herm10 (terms through t^10):")
display(Herm10)
print("Original-style exponential partial sum Hermonomial10:")
display(Hermonomial10)
print("Taylor(generator, t=0, through t^10) equals Herm10:",
      sp.simplify(generator.series(t, 0, 11).removeO() - Herm10) == 0)
print("Coefficient extraction at n=10 equals H_10:",
      sp.expand(sp.diff(generator, t, 10).subs(t, 0) - H10) == 0)

Herm10 (terms through t^10):


Original-style exponential partial sum Hermonomial10:


Taylor(generator, t=0, through t^10) equals Herm10: True


Coefficient extraction at n=10 equals H_10: True


## 4. Duplicate the final substitution/differentiation experiment

In the source cells, `s` had the displayed value $t^4-4t^3-20t^2+20t$. We reconstruct it explicitly so this notebook is self-contained, solve it, differentiate it, and substitute $H_4$ and $H_{10}$ as before.

In [6]:
s = t**4 - 4*t**3 - 20*t**2 + 20*t
roots = sp.solve(s, t)
third_derivative = sp.diff(s, t, 3)
at_H4 = sp.expand(third_derivative.subs(t, engine.get(4, prefetch=False)))
second_at_H10 = sp.expand(sp.diff(s, t, 2).subs(t, engine.get(10, prefetch=False)))

print("s =", s)
print("numeric roots =", [sp.N(root, 8) for root in roots])
print("d^3s/dt^3 =", third_derivative)
print("(d^3s/dt^3)|t=H4 =", at_H4)
print("simplified (d^2s/dt^2)|t=H10 =")
display(second_at_H10)

s = t**4 - 4*t**3 - 20*t**2 + 20*t
numeric roots = [0, 0.87934747 + 0.e-15*I, -3.4575175 - 0.e-15*I, 6.57817 - 0.e-13*I]
d^3s/dt^3 = 24*(t - 1)
(d^3s/dt^3)|t=H4 = 384*x**4 - 1152*x**2 + 264
simplified (d^2s/dt^2)|t=H10 =


## 5. Exact high-order scaling

For symbolic Hermite polynomials, a deep neural network would add training cost and approximation error to a sequence whose governing recurrence is already known. The recurrence is therefore the efficient production predictor: after $H_{n-1}$ and $H_n$ are cached, $H_{n+1}$ costs one polynomial recurrence step. We demonstrate order 100 without printing its 51 large coefficients.

In [7]:
start = perf_counter()
H100 = engine.get(100)
elapsed = perf_counter() - start
p100 = sp.Poly(H100, x)
print(f"H_100 computed in {elapsed:.4f} s")
print(f"degree={p100.degree()}, nonzero coefficients={len(p100.terms())}")
print("leading coefficient correct:", p100.LC() == 2**100)
print("parity correct:", sp.expand(H100.subs(x, -x) - (-1)**100*H100) == 0)
print("cache now extends through order", max(engine._cache))

H_100 computed in 0.6149 s
degree=100, nonzero coefficients=51
leading coefficient correct: True
parity correct: True
cache now extends through order 103


## 6. Online speculative predictor with uncertainty and correction

To model the subjective “next one, two, or three” experience, the class below observes exact normalized values

$$y_n(x_0)=\frac{H_n(x_0)}{\sqrt{2^n n!}}$$

at a selected probe point $x_0$. It fits an online autoregression to recent values, bootstraps residuals to form an empirical interval, predicts several steps recursively, then compares each forecast with the exact cached sequence. Forecasts are **hints**, never replacements for exact polynomials. Each scheduled rerun can extend the training frontier and repeat correction.

In [8]:
@dataclass
class OnlineHermiteForecaster:
    engine: HermiteLookAhead
    x0: float = 0.7
    lags: int = 6
    ridge: float = 1e-6
    seed: int = 7

    def normalized_exact(self, n: int) -> float:
        value = float(self.engine.get(n, prefetch=False).subs(x, self.x0))
        # log-domain scale avoids integer overflow before conversion at high n.
        scale = np.exp(0.5 * (n*np.log(2.0) + float(sp.loggamma(n+1))))
        return value / scale

    def _fit(self, through: int):
        y = np.array([self.normalized_exact(n) for n in range(through+1)])
        X = np.array([[1.0, *y[i-self.lags:i][::-1]] for i in range(self.lags, len(y))])
        target = y[self.lags:]
        penalty = np.eye(X.shape[1]); penalty[0, 0] = 0
        beta = np.linalg.solve(X.T @ X + self.ridge*penalty, X.T @ target)
        residuals = target - X @ beta
        return y, beta, residuals

    def forecast(self, through: int, steps: int = 3, samples: int = 1000):
        if through < self.lags + 3:
            raise ValueError("training frontier is too short for the selected lag")
        y, beta, residuals = self._fit(through)
        rng = np.random.default_rng(self.seed + through)
        paths = np.empty((samples, steps))
        for b in range(samples):
            history = list(y)
            for j in range(steps):
                row = np.array([1.0, *history[-self.lags:][::-1]])
                noise = rng.choice(residuals) if len(residuals) else 0.0
                history.append(float(row @ beta + noise))
                paths[b, j] = history[-1]
        point = np.median(paths, axis=0)
        lo, hi = np.quantile(paths, [0.025, 0.975], axis=0)
        exact = np.array([self.normalized_exact(through+j) for j in range(1, steps+1)])
        return [{"n": through+j+1, "prediction": point[j], "lower_95": lo[j],
                 "upper_95": hi[j], "exact": exact[j], "abs_error": abs(point[j]-exact[j]),
                 "covered": bool(lo[j] <= exact[j] <= hi[j])} for j in range(steps)]

forecaster = OnlineHermiteForecaster(engine, x0=0.7, lags=6)
forecast_rows = forecaster.forecast(through=30, steps=3)
for row in forecast_rows:
    print(row)

{'n': 31, 'prediction': np.float64(0.32357970201847974), 'lower_95': np.float64(0.3216673390183514), 'upper_95': np.float64(0.32712217521496656), 'exact': np.float64(0.32328500255456744), 'abs_error': np.float64(0.00029469946391230684), 'covered': True}
{'n': 32, 'prediction': np.float64(0.38527087745570854), 'lower_95': np.float64(0.3828905672310766), 'upper_95': np.float64(0.38867907829249126), 'exact': np.float64(0.38226411581651987), 'abs_error': np.float64(0.003006761639188671), 'covered': False}
{'n': 33, 'prediction': np.float64(-0.2505780187254051), 'lower_95': np.float64(-0.2556256928620685), 'upper_95': np.float64(-0.24610338811662777), 'exact': np.float64(-0.2524742198563398), 'abs_error': np.float64(0.0018962011309346938), 'covered': True}


### Continuous-learning simulation

Each cycle moves the known exact frontier forward. This is what a daily/weekly scheduled execution would do: load the cache, forecast beyond it, calculate the truth by recurrence, record errors, and save the corrected cache. Intervals here quantify bootstrap forecast dispersion, not mathematical guarantees.

In [9]:
history = []
for frontier in (20, 25, 30, 35, 40):
    rows = forecaster.forecast(through=frontier, steps=3, samples=500)
    history.append({
        "frontier": frontier,
        "mean_abs_error": float(np.mean([r["abs_error"] for r in rows])),
        "coverage": float(np.mean([r["covered"] for r in rows]))
    })
    # Exact correction becomes trusted cache; later cycles train on it.
    engine._extend_to(frontier + 3)

for row in history:
    print(row)
engine.save()
print("Saved corrected exact cache to", engine.cache_path.resolve())

{'frontier': 20, 'mean_abs_error': 0.0030190965662531075, 'coverage': 0.0}
{'frontier': 25, 'mean_abs_error': 0.0031795148712584855, 'coverage': 0.6666666666666666}
{'frontier': 30, 'mean_abs_error': 0.0016740676301938984, 'coverage': 0.6666666666666666}
{'frontier': 35, 'mean_abs_error': 0.0039049515624810563, 'coverage': 0.0}
{'frontier': 40, 'mean_abs_error': 0.0028731643343368634, 'coverage': 0.6666666666666666}


Saved corrected exact cache to /Users/brad/Documents/ChatGPT/JupyterNotebook Evals/hermite_polynomial_cache.json


## 7. Validation suite

The tests cross-check recurrence output against Rodrigues differentiation, SymPy's built-in physicists' Hermite implementation, parity, the differential equation, and the generating function. Tests are intentionally exact: a speculative forecast cannot pass into the trusted cache merely because it is close.

In [10]:
def run_tests(engine: HermiteLookAhead) -> dict[str, bool]:
    checks = {}
    checks["Rodrigues n=0..12"] = all(
        sp.expand(engine.get(n, prefetch=False) - rodrigues(n)) == 0 for n in range(13))
    checks["SymPy hermite n=0..30"] = all(
        sp.expand(engine.get(n, prefetch=False) - sp.hermite(n, x)) == 0 for n in range(31))
    checks["parity n=0..30"] = all(
        sp.expand(engine.get(n, False).subs(x, -x) - (-1)**n*engine.get(n, False)) == 0
        for n in range(31))
    checks["ODE n=0..20"] = all(
        sp.expand(sp.diff(engine.get(n, False), x, 2) - 2*x*sp.diff(engine.get(n, False), x)
                  + 2*n*engine.get(n, False)) == 0 for n in range(21))
    checks["generator through t^15"] = (
        sp.expand(engine.generator_series(15) - sp.series(sp.exp(2*x*t-t**2), t, 0, 16).removeO()) == 0)
    checks["invalid order rejected"] = False
    try:
        engine.get(-1)
    except ValueError:
        checks["invalid order rejected"] = True
    return checks

checks = run_tests(engine)
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert all(checks.values())
print(f"All {len(checks)} validation groups passed.")

PASS  Rodrigues n=0..12
PASS  SymPy hermite n=0..30
PASS  parity n=0..30
PASS  ODE n=0..20
PASS  generator through t^15
PASS  invalid order rejected
All 6 validation groups passed.


## 8. Scheduling pattern

The notebook is idempotent: rerunning it reloads `hermite_polynomial_cache.json`, extends the exact frontier, validates predictions, and writes the corrected cache. For a real scheduler, parameterize `target_order` (for example, increase it by 10 per run), execute the notebook with a tool such as Papermill or Jupyter's executor, and retain both the executed notebook and cache as run artifacts. Use atomic/versioned storage if runs may overlap.

Suggested production refinements:

- Store telemetry and forecast diagnostics in SQLite/Parquet rather than only printed output.
- Add a lock so two scheduled runs cannot update the same cache concurrently.
- Alert on failed exact tests or a sharp rise in forecast error.
- Keep recurrence-generated polynomials authoritative; use the learned model for prefetch priority or numerical intuition only.